# t-SNE on Oil Production Data
## Machine learning for energy applications — Module 4, Exercise 2

**t-Distributed Stochastic Neighbor Embedding (t-SNE)** is a non-linear
dimensionality-reduction technique that excels at visualizing high-dimensional
data in 2D or 3D. Unlike PCA, t-SNE focuses on preserving *local* neighbourhood
structure rather than global variance.

In this exercise we:
1. Explore and standardize oil production data
2. Apply t-SNE and visualize the 2D embedding
3. Experiment with the perplexity hyperparameter and with two random seeds
4. Compare t-SNE with PCA on the same data
5. Score every map with **trustworthiness** at k = 10 neighbours

> **Exploratory analysis.** The scaler and every map are fitted on all 725 wells: the maps
> describe this table. scikit-learn's `TSNE` has no `transform()` for new rows, so a map cannot
> be reused for new wells, and an island in a map is not evidence of a physical population until
> it is checked in the original features.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.manifold import TSNE, trustworthiness
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

The dataset describes oil-production wells with geophysical and petrophysical measurements:
- **md (ft)** / **tvd (ft)** — measured depth and true vertical depth
- **well spacing** — distance to nearest well
- **shale footage** — thickness of shale interval
- **acoustic impedance** — product of density and velocity
- **log permeability** / **porosity** — reservoir quality indicators
- **poisson ratio** / **p-velocity** / **s-velocity** — elastic properties

In [ ]:
# YOUR CODE HERE
# TODO: Load "Oil_production.csv" into a DataFrame
# TODO: Print the shape
# TODO: Display the first 10 rows


In [ ]:
# YOUR CODE HERE
# TODO: Call df.describe()


In [ ]:
# YOUR CODE HERE
# TODO: Call df.info()


In [ ]:
# YOUR CODE HERE
# TODO: Compute the correlation matrix and plot a heatmap
# Hint: use sns.heatmap with annot=True, cmap="coolwarm", center=0


---
## 2 — Standardize Features

t-SNE, like PCA, benefits from standardized input because it computes distances
between points in the high-dimensional space. Without scaling, the depths and footages
(in feet) would dominate every distance. The scaler is fitted on all 725 rows because the maps
are exploratory.

In [ ]:
# YOUR CODE HERE
# TODO: Get all column names as feature_cols
# TODO: Extract feature values and standardize with StandardScaler
# TODO: Print the number of features and samples


---
## 3 — Apply t-SNE with Default Perplexity

**Perplexity** is the key hyperparameter for t-SNE. It roughly corresponds to
the effective number of neighbours each point considers. Typical values range
from 5 to 50.

We also compute **trustworthiness** at k = 10: it penalizes points that are among the 10 nearest
neighbours in the map but were not neighbours in the original space (1 = no such intruders).

In [ ]:
# YOUR CODE HERE
# TODO: Create a TSNE object with n_components=2, perplexity=30, random_state=RANDOM_STATE, max_iter=1000
# TODO: Fit and transform X_scaled
# TODO: Print the output shape and KL divergence (tsne.kl_divergence_)
# TODO: Print the trustworthiness: trustworthiness(X_scaled, embedding, n_neighbors=10)

### 3.1 Visualize 2D Embedding (Coloured by Porosity)

In [ ]:
# YOUR CODE HERE
# TODO: Create a scatter plot of the t-SNE embedding
# TODO: Colour by df["porosity"] using cmap="viridis"
# TODO: Add a colorbar and label axes


---
## 4 — Experiment with Different Perplexity Values

The perplexity parameter has a significant effect on the resulting embedding:
- **Low perplexity** (5): focuses on very local structure, may produce many small clusters
- **High perplexity** (50): considers broader neighbourhoods, tends to produce more global structure

There is no single "best" perplexity — it depends on the data and the analysis goal.
Do not compare KL divergence across perplexities (each perplexity defines a different target);
compare the maps with trustworthiness at the same k instead.

In [ ]:
# YOUR CODE HERE
# TODO: For perplexities = [5, 15, 30, 50]:
#   - Fit t-SNE (same random_state, max_iter=1000) and store the embeddings in a dictionary
#   - Compute the trustworthiness (n_neighbors=10) of each map and store it in a second dictionary
#   - Print the KL divergence and the trustworthiness for each

### 4.1 Perplexity Comparison (2x2 Subplot)

In [ ]:
# YOUR CODE HERE
# TODO: Create a 2x2 subplot figure
# TODO: For each perplexity, scatter the embedding coloured by porosity;
#       put the perplexity and the trustworthiness in the subplot title
# TODO: Add a shared colorbar
# Hint: call plt.tight_layout(rect=[0, 0, 0.89, 1]) first, then
#       fig.add_axes([x, y, w, h]) for a manual colorbar axis

### 4.2 Two Random Seeds

With the default `init="pca"` the layout barely depends on `random_state`. To see how much a map
depends on its random start, use `init="random"` with two seeds, at a small and a large
perplexity. Compare what changes between the maps (arrangement and orientation of the groups)
with what does not (trustworthiness, local neighbourhoods).

In [ ]:
# YOUR CODE HERE
# TODO: For seeds 0 and 1 and perplexities 5 and 50, fit
#       TSNE(n_components=2, perplexity=p, init="random", random_state=seed, max_iter=1000)
# TODO: Compute trustworthiness(X_scaled, embedding, n_neighbors=10) for each map
# TODO: Plot the four maps in a 2x2 grid (coloured by porosity) with seed, perplexity and
#       trustworthiness in each title. What changed between the seeds, and what did not?

---
## 5 — Colouring by Different Features

It is helpful to explore the embedding coloured by different features to understand
what structure t-SNE has captured. These features are inputs of the map, so the colours show
which inputs vary smoothly across it; they are not independent evidence of groups.

In [ ]:
# YOUR CODE HERE
# TODO: Choose 4 features: "porosity", "log permeability", "acoustic impedance", "poisson ratio"
# TODO: Create a 2x2 subplot using the perplexity=30 embedding
# TODO: Colour each subplot by a different feature


---
## 6 — Compare t-SNE with PCA

PCA and t-SNE optimize for different objectives:
- **PCA**: maximizes global variance; preserves large pairwise distances
- **t-SNE**: preserves local neighbourhoods; can distort global distances

Let us compare the two side by side.

In [ ]:
# YOUR CODE HERE
# TODO: Fit PCA with n_components=2 on X_scaled
# TODO: Print the explained variance ratio for each component and the total
# TODO: Print the trustworthiness (n_neighbors=10) of the PCA map next to the t-SNE one

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x2 subplot figure
#   Left: PCA scatter coloured by porosity with variance labels on axes
#   Right: t-SNE scatter (perplexity=30) coloured by porosity
# TODO: Put the trustworthiness of each map in its title
# TODO: Add colorbars to each subplot

---
## 7 — Limitations and When to Use t-SNE

### Strengths
- Excellent for **visual exploration** of high-dimensional data
- Reveals clusters and local structure that linear methods like PCA miss
- Works well when the data lies on a non-linear manifold

### Limitations
- **Seed-dependent** — with `init="random"`, different seeds give different layouts; compare at least two
- **Perplexity-sensitive** — results change significantly with different perplexity values
- **No transform for new points** — scikit-learn's `TSNE` has no `transform()`; new wells cannot be placed on an existing map without refitting
- **Does not preserve global structure** — distances between clusters, and cluster sizes, in the plot may be meaningless
- **Computationally expensive** — O(n^2) for the exact algorithm; Barnes-Hut approximation helps
- **Not suitable for feature extraction** — the embedding dimensions have no interpretable meaning

### When to use t-SNE
- Exploring structure in high-dimensional data as a **visualization tool**
- When you suspect non-linear relationships that PCA would miss
- For a first look at possible groups before clustering; maps can exaggerate separation, so
  confirm any group in the original features

---
## Summary

In this exercise we applied t-SNE to oil production data and covered:

1. **Data preparation** — standardization is important for distance-based methods.
2. **t-SNE embedding** — projecting 10 features into 2D for visualization.
3. **Perplexity and seeds** — low perplexity captures local detail; high perplexity reveals broader patterns; with `init="random"` the layout changes with the seed.
4. **Trustworthiness** — a neighbourhood score at a stated k, comparable across settings, seeds and methods.
5. **Feature colouring** — exploring which features vary across the embedding.
6. **PCA comparison** — PCA preserves global variance while t-SNE preserves local neighbourhoods.
7. **Limitations** — t-SNE is a visualization tool, not a general-purpose dimensionality reduction method.

**Key takeaway**: t-SNE is a powerful tool for exploring high-dimensional oil and gas
data, but its results should be interpreted carefully. Always try multiple perplexity
values and at least two seeds, report the settings and the trustworthiness with every map,
and use it alongside PCA for a complete picture.